# zagg write pipeline

Lambda fan-out from the shard maps built in `01_query`: dispatch, futures, cost. Same shardmap, different aggregations; with and without the strict-AOI mask; California at o8 with cost telemetry.

In [ ]:
# %pip install "zagg[catalog,viz]"

import io
import json
import logging
import os
import re
import time
from importlib import resources
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import Markdown

from zagg.catalog import load_polygon
from zagg.catalog.shardmap import ShardMap
from zagg.catalog.sources import Catalog
from zagg.client import Run
from zagg.config import default_config
from zagg.data import demo_aoi
from zagg.grids import from_config
from zagg.notebook import format_max_cost, max_cost_preview

In [ ]:
os.environ.setdefault("AWS_PROFILE", "nasa")
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
for noisy in ("botocore", "boto3", "urllib3", "s3transfer", "stac_geoparquet"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

OUT = Path("outputs")
STORE = "s3://sliderule-public/zagg-demo"

timings = {}


class stage:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        timings[self.name] = round(time.perf_counter() - self.t0, 2)
        print(f"[{self.name}] {timings[self.name]:.1f}s")

## The aggregation template, block by block

In [ ]:
raw = (resources.files("zagg.configs") / "atl03_tdigest_healpix_hive.yaml").read_text()


def yaml_block(text, name):
    block = re.search(rf"^{name}:.*?(?=^\w|\Z)", text, re.S | re.M).group(0)
    clean = "\n".join(l for l in block.splitlines() if not l.strip().startswith("#"))
    return Markdown(f"```yaml\n{clean}\n```")

### `data_source` — what to read

- h5coro reads each ATL03 granule directly over S3 by byte range — nothing is downloaded; six beam groups each contribute photon lat/lon/height.
- `filters` drops TEP returns; `levels` + `read_plan` declare the segment→photon index link, so a worker fetches only the segments crossing its shard (`pad: 1`).

In [ ]:
yaml_block(raw, "data_source")

### `aggregation` — what to compute

- One output field per entry: `count` is a dense per-cell int32; `h_tdigest` is a ragged per-cell t-digest sketch (`delta: 256` centroid budget) from which any quantile is recovered at read time.
- Swapping this block changes the science without touching the read or the layout — demoed below with the located variant.

In [ ]:
yaml_block(raw, "aggregation")

### `output` — where it lands

- HEALPix nested grid: `child_order: 19` (~10 m cells); `parent_order: 9` is the dispatch shard (1024×1024 cells, one Lambda invoke each); `chunk_inner: 13` bundles 256 inner chunks into one sharded zarr object.
- `store_layout: hive`: every shard writes its own self-describing leaf zarr; `pyramid: false` keeps the overview sweep off.

In [ ]:
yaml_block(raw, "output")

## SERC — dispatch with futures

In [ ]:
serc_map = str(OUT / "shardmap_atl03_serc_o9.json")
config = default_config("atl03_tdigest_healpix_hive")

run = Run.from_config(config, shardmap=serc_map, store=f"{STORE}/serc_tdigest.zarr", overwrite=True)
run

In [ ]:
handle = run.dispatch()  # returns after the setup handshake; every shard is a Future
handle.status()

In [ ]:
with stage("SERC: t-digest run"):
    for fut in handle.progress():
        pass
handle.status()

In [ ]:
res = next(iter(handle.futures.values())).result()
{k: res.get(k) for k in ("shard_key", "wall_time", "lambda_duration", "retries")} | {
    k: v for k, v in res["body"].items() if isinstance(v, (int, float, str))
}

## SERC — same pipeline, with the strict-AOI mask

In [ ]:
cat_serc = Catalog.from_geoparquet(str(OUT / "catalog_atl03_serc.parquet"))
serc_parts = load_polygon(demo_aoi("serc"))

masked_config = default_config("atl03_tdigest_healpix_hive")
masked_config.output["aoi_mask"] = True
grid = from_config(masked_config)

sm_masked = ShardMap.build(cat_serc, grid, region=serc_parts, mortie_order=9)
sm_masked.to_json(str(OUT / "shardmap_atl03_serc_o9_aoi.json"))

# in-AOI cell fraction per shard: edge shards are mostly outside the flight box
{
    grid.shard_label(int(k)): round(float(grid.aoi_mask_from_payload(m, grid.children(int(k))).mean()), 3)
    for k, m in zip(sm_masked.shard_keys, sm_masked.aoi_mask)
}

In [ ]:
run_masked = Run.from_config(
    masked_config,
    shardmap=str(OUT / "shardmap_atl03_serc_o9_aoi.json"),
    store=f"{STORE}/serc_tdigest_aoi.zarr",
    overwrite=True,
)
with stage("SERC: t-digest + AOI mask"):
    handle_masked = run_masked.dispatch()
    for fut in handle_masked.progress():
        pass

In [ ]:
def rollup(h):
    bodies = [f.result().get("body", {}) for f in h.futures.values()]
    return {
        "cells_with_data": sum(b.get("cells_with_data") or 0 for b in bodies),
        "total_obs": sum(b.get("total_obs") or 0 for b in bodies),
    }


pd.DataFrame({"unmasked": rollup(handle), "aoi_masked": rollup(handle_masked)})

The mask drops no photons — the masked store additionally carries a per-cell `aoi_mask` bool array, which readers use to clip whole-shard overhang at the AOI edge.

## SERC — same shardmap, different aggregation: located t-digest

Only the `aggregation` block changes — each t-digest centroid gains a high-resolution morton location channel:

In [ ]:
located_config = default_config("atl03_tdigest_healpix_hive")
located_config.aggregation = default_config("atl03_tdigest_located_healpix").aggregation

located_yaml = (resources.files("zagg.configs") / "atl03_tdigest_located_healpix.yaml").read_text()
yaml_block(located_yaml, "aggregation")

In [ ]:
run_located = Run.from_config(
    located_config, shardmap=serc_map, store=f"{STORE}/serc_tdigest_located.zarr", overwrite=True
)
with stage("SERC: located t-digest run"):
    handle_located = run_located.dispatch()
    for fut in handle_located.progress():
        pass
handle_located.status()

## California at o8 — cost ceiling, first-shard latency, actuals

In [ ]:
ca_config = default_config("atl03_tdigest_healpix_hive")
ca_config.output["grid"]["parent_order"] = 8
ca_map = str(OUT / "shardmap_california_o8.json")

print(format_max_cost(max_cost_preview(ca_config, catalog=ca_map)))

In [ ]:
run_ca = Run.from_config(
    ca_config, shardmap=ca_map, store=f"{STORE}/california_tdigest_o8.zarr", overwrite=True
)

t0 = time.perf_counter()
handle_ca = run_ca.dispatch()
completions = []
with stage("California: o8 run"):
    for fut in handle_ca.progress():
        completions.append(time.perf_counter() - t0)
print(
    f"first shard done at {completions[0]:.1f}s; "
    f"full state at {completions[-1]:.1f}s ({len(completions)} shards)"
)

Per-shard telemetry lands as a parquet at the store root (one row per shard: durations, phase timings, observation counts, priced GB-seconds).

In [ ]:
def fetch_run_stats(store_path):
    bucket, _, prefix = store_path.removeprefix("s3://").partition("/")
    s3 = boto3.client("s3")
    objs = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/stats_")["Contents"]
    key = max(o["Key"] for o in objs)  # timestamp-first names sort chronologically
    return pq.read_table(io.BytesIO(s3.get_object(Bucket=bucket, Key=key)["Body"].read())).to_pandas()


stats = fetch_run_stats(handle_ca.store_path)
print(
    f"actual cost ${stats.est_cost_usd.sum():.2f} across {len(stats)} shards; "
    f"{stats.duration_s.sum():,.0f} lambda-seconds, {stats.n_obs.sum() / 1e9:.2f}B photons"
)
stats[["duration_s", "n_obs", "n_granules", "gb_seconds", "est_cost_usd"]].describe().round(4)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

t = np.asarray(completions)
n = np.arange(1, len(t) + 1)
ax1.step(t, n, where="post", color="#4269d0", lw=2)
ax1.axvline(t[0], color="#9498a0", lw=1, ls="--")
ax1.annotate(f"first shard {t[0]:.0f}s", (t[0], len(t)), xytext=(6, -2), textcoords="offset points", fontsize=9, color="#555")
ax1.annotate(f"wall {t[-1]:.0f}s", (t[-1], len(t) * 0.6), xytext=(-6, 0), textcoords="offset points", ha="right", fontsize=9, color="#555")
ax1.set_xlabel("seconds since dispatch")
ax1.set_ylabel("shards complete")
ax1.set_title("Fleet completion — California o8", fontsize=11)

ax2.scatter(stats.n_obs / 1e6, stats.est_cost_usd * 100, s=12, alpha=0.45, color="#4269d0", edgecolors="none")
ax2.set_xlabel("photons aggregated (millions)")
ax2.set_ylabel("shard cost (¢)")
ax2.set_title("Per-shard cost vs photon volume", fontsize=11)

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.25, lw=0.5)
plt.tight_layout()

## Timings

In [ ]:
(OUT / "timings_write.json").write_text(json.dumps(timings, indent=2))
pd.Series(timings, name="seconds").to_frame()